### Required Libraries here

In [1]:
import cv2 
import os 
import numpy as np 
from matplotlib import pyplot as plt    
import torch  
import torch.nn as nn  
from torch.utils.data import Dataset, DataLoader 
import random 
import torchvision.transforms as transforms

In [3]:
class ASLDataset(Dataset):
    def __init__(self, image_paths, labels, augment=False):
        self.image_paths = image_paths
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def augment_image(self, img):
        angle = random.uniform(-15, 15)
        h, w = img.shape
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
        img = cv2.warpAffine(img, M, (w, h))

        alpha = random.uniform(0.8, 1.2)
        img = np.clip(img * alpha, 0, 255)

        if random.random() < 0.3:
            img = cv2.GaussianBlur(img, (3,3), 0)

        return img

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx], cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64, 64))

        if self.augment:
            img = self.augment_image(img)

        img = img.astype(np.float32) / 255.0
        img = (img - img.mean()) / (img.std() + 1e-6)

        img = torch.tensor(img).unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label


### Loading the data 

In [4]:
ds="image" #path to the dataset 
x=[] 
y=[] 
#mapping of folders (labels-> numbers) 
label_map={ 
        "A": 0,
        "B": 1,
        "C": 2,
        "D": 3,
        "E": 4,
        "F": 5,
        "G": 6,
        "H": 7,
        "I": 8,
        "J": 9,
        "K": 10,
        "L": 11,
        "M": 12,
        "N": 13,
        "O": 14,
        "P": 15,
        "Q": 16,
        "R": 17,
        "S": 18,
        "T": 19,
        "U": 20,
        "V": 21,
        "W": 22,
        "X": 23,
        "Y": 24,
        "Z": 25, 
        "nothing": 26, 
        "del": 27, 
        "space": 28    
} 

In [5]:
for label in sorted(os.listdir(ds)):
    label_folder = os.path.join(ds, label)
    for img_name in os.listdir(label_folder):
        x.append(os.path.join(label_folder, img_name))
        y.append(label_map[label])

In [ ]:
# from torchvision import transforms
# transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(15),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
#     transforms.ToTensor()
# ])

### Creating dataset and loader here

In [6]:
dataset = ASLDataset(x, y, augment=True)

In [7]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [8]:
class ASL_CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Dropout(0.4)
        )

        self.classifier = nn.Sequential(
            nn.Linear(32 * 16 * 16, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [9]:
num_classes = len(label_map)  # automatically 29
model = ASL_CNN(num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
print("Model output classes:", model.classifier[-1].out_features)
print("Max label:", max(y)) 

NameError: name 'device' is not defined

In [57]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

Epoch [1/5] Loss: 2.2684, Accuracy: 0.2795
Epoch [2/5] Loss: 1.6881, Accuracy: 0.4250
Epoch [3/5] Loss: 1.4995, Accuracy: 0.4863
Epoch [4/5] Loss: 1.3806, Accuracy: 0.5262
Epoch [5/5] Loss: 1.3057, Accuracy: 0.5509


In [40]:
# model.eval()
# val_correct, val_total = 0, 0
# with torch.no_grad():
#         for val_images, val_labels in val_loader:
#             val_images, val_labels = val_images.to(device), val_labels.to(device)
#             val_outputs = model(val_images)
#             _, val_predicted = torch.max(val_outputs, 1)
#             val_correct += (val_predicted == val_labels).sum().item()
#             val_total += val_labels.size(0)
# val_acc = val_correct / val_total
# model.train()

# print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, " f"Train Acc: {epoch_acc:.4f}, Val Acc: {val_acc:.4f}") 
# torch.save(model.state_dict(), "asl_cnn.pth")

In [60]:
model.eval()
inv_label_map = {v: k for k, v in label_map.items()} 
probs = torch.softmax(outputs, dim=1)
conf, pred = probs.max(1)

labels = []
for c, p in zip(conf, pred):
    if c.item() < 0.4:
        labels.append("Unknown")
    else:
        labels.append(inv_label_map[p.item()])

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (64, 64))
    gray = gray.astype(np.float32) / 255.0
    gray = (gray - gray.mean()) / (gray.std() + 1e-6)

    img = torch.tensor(gray).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = probs.max(1)

    if conf.item() < 0.6:
        label = "Unknown"
    else:
        label = inv_label_map[pred.item()]

    cv2.putText(frame, f"{label} ({conf.item():.2f})",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.imshow("ASL Prediction", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [18]:
dataset_path = "asl_dataset1"  # change if needed

image_paths = []
labels = []
label_map = {}

for idx, cls in enumerate(sorted(os.listdir(dataset_path))):
    label_map[cls] = idx
    cls_path = os.path.join(dataset_path, cls)

    imgs = os.listdir(cls_path)[:20]  # take only 2 images per class
    for img in imgs:
        image_paths.append(os.path.join(cls_path, img))
        labels.append(idx)

print("Total images:", len(image_paths))
print("Label map:", label_map)


Total images: 720
Label map: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, 'a': 10, 'b': 11, 'c': 12, 'd': 13, 'e': 14, 'f': 15, 'g': 16, 'h': 17, 'i': 18, 'j': 19, 'k': 20, 'l': 21, 'm': 22, 'n': 23, 'o': 24, 'p': 25, 'q': 26, 'r': 27, 's': 28, 't': 29, 'u': 30, 'v': 31, 'w': 32, 'x': 33, 'y': 34, 'z': 35}


In [19]:
class ASLDataset(Dataset):
    def __init__(self, image_paths, labels, train=True):
        self.image_paths = image_paths
        self.labels = labels
        self.train = train

        self.augment = transforms.RandomApply([
            transforms.RandomRotation(10),
            transforms.RandomAffine(0, translate=(0.05, 0.05))
        ], p=0.5)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx], cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64, 64))
        img = img.astype(np.float32) / 255.0

        # Per-image normalization (VERY IMPORTANT)
        img = (img - img.mean()) / (img.std() + 1e-6)

        img = torch.tensor(img).unsqueeze(0)  # [1, 64, 64]

        if self.train:
            img = self.augment(img)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label


In [20]:
train_dataset = ASLDataset(image_paths, labels, train=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)


In [21]:
class ASL_CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(32 * 16 * 16, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # 64 → 32
        x = self.pool(torch.relu(self.conv2(x)))  # 32 → 16
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ASL_CNN(num_classes=len(label_map)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    correct = total = 0
    loss_sum = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1} | Loss {loss_sum:.3f} | Acc {correct/total:.2f}")


Epoch 1 | Loss 240.058 | Acc 0.30
Epoch 2 | Loss 75.602 | Acc 0.75
Epoch 3 | Loss 52.795 | Acc 0.82
Epoch 4 | Loss 34.891 | Acc 0.88
Epoch 5 | Loss 28.218 | Acc 0.90
Epoch 6 | Loss 26.304 | Acc 0.91
Epoch 7 | Loss 21.187 | Acc 0.92
Epoch 8 | Loss 16.594 | Acc 0.93
Epoch 9 | Loss 20.543 | Acc 0.93
Epoch 10 | Loss 14.223 | Acc 0.95


In [23]:
model.eval()


ASL_CNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=8192, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=36, bias=True)
)

In [24]:
import cv2
import torch
import numpy as np

inv_label_map = {v: k for k, v in label_map.items()}

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---- PREPROCESS (MATCH TRAINING) ----
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Crop center region (VERY IMPORTANT)
    h, w = gray.shape
    size = min(h, w)
    start_x = w//2 - size//4
    start_y = h//2 - size//4
    crop = gray[start_y:start_y+size//2, start_x:start_x+size//2]

    img = cv2.resize(crop, (64, 64))
    img = img.astype(np.float32) / 255.0
    img = (img - img.mean()) / (img.std() + 1e-6)

    img = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)

    # ---- PREDICTION ----
    with torch.no_grad():
        outputs = model(img)
        pred = outputs.argmax(1).item()
        label = inv_label_map[pred]

    # ---- DISPLAY ----
    cv2.rectangle(frame,
                  (start_x, start_y),
                  (start_x + size//2, start_y + size//2),
                  (0, 255, 0), 2)

    cv2.putText(frame, f"Predicted: {label}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.imshow("ASL Webcam Prediction", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
